<a href="https://colab.research.google.com/github/gonzH/ifes-redes-neurais-artificiais/blob/main/Trab02_MNIST_Convolucao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Prencha suas informações: { display-mode: "form" }

nome_completo = 'Hellesandro Gonzaga de Carvalho'  # @param {type: "string"}
matricula     = '20251MCPA0110'  # @param {type: "string"}

data = '2026-07-03'  # @param {type: "date"}
# @markdown ---

## Tarefa:

> Procurar a base de dados original do MNIST e rodar uma rede neural com camadas convolucionais.

> Não pode ser a que vem no Keras. Tem que baixar e fazer o preprocessamento.

## Observação:
A base de dados original do MNIST estava vazia na data 04/07/2026 no endereço (http://yann.lecun.com/exdb/mnist/), então peguei uma "original" do HuggingFace (https://www.kaggle.com/datasets/avnishnish/mnist-original).


# Download

In [1]:
# API Legacy do Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# download
!kaggle datasets download -d avnishnish/mnist-original

# unzip
!unzip -o mnist-original.zip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/avnishnish/mnist-original
License(s): CC0-1.0
100% 10.9M/10.9M [00:01<00:00, 7.12MB/s]

Archive:  mnist-original.zip
  inflating: mnist-original.mat      


# Pré-processamento

In [2]:
import numpy as np
import scipy.io
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

mat_data = scipy.io.loadmat("mnist-original.mat")

# coluna/linha data (784, 70000), labels em (1, 70000)
raw_data = mat_data["data"]
raw_labels = mat_data["label"]

# passa para linha/coluna o data (70000, 784) e labels: (70000,)
X = raw_data.T
y = raw_labels.T.squeeze().astype(np.int32)

# o MNIST armazena cada imagem de 28x28 como um vetor linear de 784 pixels.
# reshape para matrizes 28x28 com 1 canal de cor (escala de cinza)
X = X.reshape(-1, 28, 28, 1)

# normaliza os pixels para a escala [0, 1]
X = X.astype("float32") / 255.0

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=10000, random_state=42, stratify=y
)

print(f"Shape do Treino: {x_train.shape} | Labels: {y_train.shape}")
print(f"Shape do Teste:  {x_test.shape} | Labels: {y_test.shape}\n")

Shape do Treino: (60000, 28, 28, 1) | Labels: (60000,)
Shape do Teste:  (10000, 28, 28, 1) | Labels: (10000,)



# CNN e Treino/Teste

In [3]:
# CNN
model = models.Sequential(
    [
        layers.Conv2D(32, (3, 3), activation="relu", input_shape=(28, 28, 1)),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(
            10, activation="softmax"
        ),  # 10 neurônios de saída, dígitos 0 a 9
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
print("Treinando a CNN...")
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.1)

test_loss, test_acc = model.evaluate(x_test, y_test)
print(f"\nAcuracia: {test_acc:.4f}")

Treinando a CNN...
Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.9492 - loss: 0.1728 - val_accuracy: 0.9763 - val_loss: 0.0713
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9842 - loss: 0.0518 - val_accuracy: 0.9855 - val_loss: 0.0439
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9893 - loss: 0.0362 - val_accuracy: 0.9870 - val_loss: 0.0402
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9911 - loss: 0.0267 - val_accuracy: 0.9868 - val_loss: 0.0409
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9930 - loss: 0.0214 - val_accuracy: 0.9888 - val_loss: 0.0373
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9884 - loss: 0.0356

Acuracia: 0.9884
